# Pandas II — Limpieza de datos y construcción de una tabla analítica

## Objetivos
- Diagnosticar calidad de datos con rapidez.
- Limpiar duplicados, nulos y texto.
- Validar tipos y reglas de negocio básicas.
- Unir tablas con `merge` y construir una `fact_sales`.
- Dejar el dataset listo para análisis en Pandas III.

---
## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)


---
## 1. Datos de trabajo

Trabajaremos con tres tablas con problemas realistas:
- `customers`: dimensión de clientes
- `products`: dimensión de productos
- `transactions`: tabla de transacciones

La idea es limpiarlas y convertirlas en una tabla analítica única.

In [2]:
# --- Customers ---
n_customers = 120
customer_ids = np.arange(1000, 1000 + n_customers)

cities = np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], size=n_customers)
segments = np.random.choice(['Standard', 'Premium'], size=n_customers, p=[0.75, 0.25])
ages = np.random.normal(39, 12, size=n_customers).round().astype(float)
names = [f' customer_{i} ' for i in range(n_customers)]

customers = pd.DataFrame({
    'customer_id': customer_ids,
    'name': names,
    'city': cities,
    'segment': segments,
    'age': ages,
})

customers.loc[np.random.choice(customers.index, 10, replace=False), 'age'] = np.nan
customers.loc[np.random.choice(customers.index, 6, replace=False), 'city'] = None
customers = pd.concat([customers, customers.sample(8, random_state=42)], ignore_index=True)

# --- Products ---
product_ids = np.arange(200, 240)
categories = np.random.choice(['Electronics', 'Home', 'Sports', 'Books'], size=len(product_ids))
base_prices = np.random.uniform(5, 250, size=len(product_ids)).round(2)
product_names = [f' Product_{pid} ' for pid in product_ids]

products = pd.DataFrame({
    'product_id': product_ids,
    'product_name': product_names,
    'category': categories,
    'base_price': base_prices,
})

products.loc[np.random.choice(products.index, 5, replace=False), 'category'] = ' electronics '
products.loc[np.random.choice(products.index, 4, replace=False), 'base_price'] = np.nan
products = pd.concat([products, products.sample(5, random_state=1)], ignore_index=True)

# --- Transactions ---
n_tx = 900
tx_ids = np.arange(5000, 5000 + n_tx)

tx_customer = np.random.choice(customers['customer_id'].unique(), size=n_tx)
tx_product = np.random.choice(products['product_id'].unique(), size=n_tx)
quantity = np.random.randint(1, 6, size=n_tx)
discount = np.random.choice([0.0, 0.05, 0.10, np.nan], size=n_tx, p=[0.55, 0.2, 0.15, 0.10])

dates = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 90, size=n_tx), unit='D')

transactions = pd.DataFrame({
    'tx_id': tx_ids,
    'customer_id': tx_customer,
    'product_id': tx_product,
    'quantity': quantity,
    'discount': discount,
    'tx_date': dates,
})

transactions.loc[np.random.choice(transactions.index, 7, replace=False), 'product_id'] = np.nan
transactions = pd.concat([transactions, transactions.sample(12, random_state=7)], ignore_index=True)

print('customers', customers.shape)
print('products', products.shape)
print('transactions', transactions.shape)


customers (128, 5)
products (45, 4)
transactions (912, 6)


---
## 2. Diagnóstico inicial

Antes de tocar nada, conviene mirar:
- tamaño
- tipos
- nulos
- duplicados
- claves potencialmente problemáticas

### Ejemplo: forma, nulos y duplicados

In [ ]:
print(customers.shape)
print(products.shape)
print(transactions.shape)

display(customers.isna().sum())
display(products.isna().sum())
display(transactions.isna().sum())

print('Duplicated rows customers:', customers.duplicated().sum())
print('Duplicated rows products:', products.duplicated().sum())
print('Duplicated rows transactions:', transactions.duplicated().sum())

### Ejercicio 1
1. Ejecuta `info()` para las tres tablas.
2. Calcula duplicados completos en cada una.
3. Localiza IDs duplicados en `customer_id`, `product_id` y `tx_id`.

In [3]:
# TODO
# ==========================================
# 1. Información general de cada DataFrame
# ==========================================

print("=== CUSTOMERS INFO ===")
customers.info()

print("\n=== PRODUCTS INFO ===")
products.info()

print("\n=== TRANSACTIONS INFO ===")
transactions.info()


# ==========================================
# 2. Duplicados completos (filas idénticas)
# ==========================================

print("\n=== DUPLICADOS COMPLETOS ===")
print(f"Customers: {customers.duplicated().sum()}")
print(f"Products: {products.duplicated().sum()}")
print(f"Transactions: {transactions.duplicated().sum()}")


# ==========================================
# 3. IDs duplicados
# ==========================================

print("\n=== CUSTOMER_ID DUPLICADOS ===")
customer_dup_ids = customers[
    customers.duplicated(subset='customer_id', keep=False)
].sort_values('customer_id')
display(customer_dup_ids)

print("\nNúmero de customer_id duplicados:",
      customer_dup_ids['customer_id'].nunique())


print("\n=== PRODUCT_ID DUPLICADOS ===")
product_dup_ids = products[
    products.duplicated(subset='product_id', keep=False)
].sort_values('product_id')
display(product_dup_ids)

print("\nNúmero de product_id duplicados:",
      product_dup_ids['product_id'].nunique())


print("\n=== TX_ID DUPLICADOS ===")
tx_dup_ids = transactions[
    transactions.duplicated(subset='tx_id', keep=False)
].sort_values('tx_id')
display(tx_dup_ids)

print("\nNúmero de tx_id duplicados:",
      tx_dup_ids['tx_id'].nunique())

=== CUSTOMERS INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  128 non-null    int64  
 1   name         128 non-null    str    
 2   city         122 non-null    str    
 3   segment      128 non-null    str    
 4   age          118 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 5.1 KB

=== PRODUCTS INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    45 non-null     int64  
 1   product_name  45 non-null     str    
 2   category      45 non-null     str    
 3   base_price    40 non-null     float64
dtypes: float64(1), int64(1), str(2)
memory usage: 1.5 KB

=== TRANSACTIONS INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (

,customer_id,name,city,segment,age
4,1004,customer_4,Valencia,Standard,42.0
122,1004,customer_4,Valencia,Standard,42.0
127,1010,customer_10,Valencia,Premium,24.0
10,1010,customer_10,Valencia,Premium,24.0
26,1026,customer_26,Barcelona,Standard,53.0
124,1026,customer_26,Barcelona,Standard,53.0
120,1044,customer_44,Sevilla,Premium,47.0
44,1044,customer_44,Sevilla,Premium,47.0
121,1047,customer_47,Valencia,Premium,45.0
47,1047,customer_47,Valencia,Premium,45.0



Número de customer_id duplicados: 8

=== PRODUCT_ID DUPLICADOS ===


,product_id,product_name,category,base_price
2,202,Product_202,electronics,188.65
40,202,Product_202,electronics,188.65
42,203,Product_203,Home,101.28
3,203,Product_203,Home,101.28
21,221,Product_221,Books,66.60
43,221,Product_221,Books,66.60
44,227,Product_227,Books,70.82
27,227,Product_227,Books,70.82
41,231,Product_231,Electronics,NaN
31,231,Product_231,Electronics,NaN



Número de product_id duplicados: 5

=== TX_ID DUPLICADOS ===


,tx_id,customer_id,product_id,quantity,discount,tx_date
60,5060,1017,228.0,1,0.00,2025-03-10
908,5060,1017,228.0,1,0.00,2025-03-10
909,5063,1110,233.0,2,0.00,2025-01-05
63,5063,1110,233.0,2,0.00,2025-01-05
98,5098,1061,231.0,5,0.00,2025-03-12
910,5098,1061,231.0,5,0.00,2025-03-12
900,5256,1050,212.0,4,0.00,2025-01-09
256,5256,1050,212.0,4,0.00,2025-01-09
902,5259,1066,212.0,2,0.05,2025-03-31
259,5259,1066,212.0,2,0.05,2025-03-31



Número de tx_id duplicados: 12


---
## 3. Duplicados

Hay dos casos muy típicos:
- duplicados exactos
- duplicados por clave

Normalmente primero eliminamos duplicados completos y luego resolvemos la unicidad de las claves.

### Ejemplo: deduplicar por filas y por clave

In [ ]:
transactions_dedup = transactions.drop_duplicates()
customers_dedup = customers.drop_duplicates(subset=['customer_id'], keep='first')
products_dedup = products.drop_duplicates(subset=['product_id'], keep='first')

print('transactions:', transactions.shape, '->', transactions_dedup.shape)
print('customers:', customers.shape, '->', customers_dedup.shape)
print('products:', products.shape, '->', products_dedup.shape)

### Ejercicio 2
1. Verifica que `customer_id` y `product_id` quedan únicos tras deduplicar.
2. Comprueba que `tx_id` queda único en `transactions_dedup`.
3. Muestra las filas conflictivas antes de limpiar para justificar la decisión.

In [4]:
# TODO
# ==========================================
# 1. Mostrar filas conflictivas antes de limpiar
# ==========================================

print("=== CUSTOMER_ID DUPLICADOS (ANTES DE LIMPIAR) ===")
customer_conflicts = customers[
    customers.duplicated(subset='customer_id', keep=False)
].sort_values('customer_id')
display(customer_conflicts)

print("\n=== PRODUCT_ID DUPLICADOS (ANTES DE LIMPIAR) ===")
product_conflicts = products[
    products.duplicated(subset='product_id', keep=False)
].sort_values('product_id')
display(product_conflicts)

print("\n=== TX_ID DUPLICADOS (ANTES DE LIMPIAR) ===")
tx_conflicts = transactions[
    transactions.duplicated(subset='tx_id', keep=False)
].sort_values('tx_id')
display(tx_conflicts)


# ==========================================
# 2. Deduplicar tablas
# ==========================================

# Mantener la primera aparición de cada ID
customers_dedup = customers.drop_duplicates(subset='customer_id', keep='first')
products_dedup = products.drop_duplicates(subset='product_id', keep='first')
transactions_dedup = transactions.drop_duplicates(subset='tx_id', keep='first')


# ==========================================
# 3. Verificar unicidad de IDs
# ==========================================

print("\n=== VERIFICACIÓN DE UNICIDAD ===")

print("customer_id único:",
      customers_dedup['customer_id'].is_unique)

print("product_id único:",
      products_dedup['product_id'].is_unique)

print("tx_id único:",
      transactions_dedup['tx_id'].is_unique)


# ==========================================
# 4. Número de filas tras limpieza
# ==========================================

print("\n=== SHAPES TRAS DEDUPLICAR ===")
print("customers_dedup:", customers_dedup.shape)
print("products_dedup:", products_dedup.shape)
print("transactions_dedup:", transactions_dedup.shape)

=== CUSTOMER_ID DUPLICADOS (ANTES DE LIMPIAR) ===


,customer_id,name,city,segment,age
4,1004,customer_4,Valencia,Standard,42.0
122,1004,customer_4,Valencia,Standard,42.0
127,1010,customer_10,Valencia,Premium,24.0
10,1010,customer_10,Valencia,Premium,24.0
26,1026,customer_26,Barcelona,Standard,53.0
124,1026,customer_26,Barcelona,Standard,53.0
120,1044,customer_44,Sevilla,Premium,47.0
44,1044,customer_44,Sevilla,Premium,47.0
121,1047,customer_47,Valencia,Premium,45.0
47,1047,customer_47,Valencia,Premium,45.0



=== PRODUCT_ID DUPLICADOS (ANTES DE LIMPIAR) ===


,product_id,product_name,category,base_price
2,202,Product_202,electronics,188.65
40,202,Product_202,electronics,188.65
42,203,Product_203,Home,101.28
3,203,Product_203,Home,101.28
21,221,Product_221,Books,66.60
43,221,Product_221,Books,66.60
44,227,Product_227,Books,70.82
27,227,Product_227,Books,70.82
41,231,Product_231,Electronics,NaN
31,231,Product_231,Electronics,NaN



=== TX_ID DUPLICADOS (ANTES DE LIMPIAR) ===


,tx_id,customer_id,product_id,quantity,discount,tx_date
60,5060,1017,228.0,1,0.00,2025-03-10
908,5060,1017,228.0,1,0.00,2025-03-10
909,5063,1110,233.0,2,0.00,2025-01-05
63,5063,1110,233.0,2,0.00,2025-01-05
98,5098,1061,231.0,5,0.00,2025-03-12
910,5098,1061,231.0,5,0.00,2025-03-12
900,5256,1050,212.0,4,0.00,2025-01-09
256,5256,1050,212.0,4,0.00,2025-01-09
902,5259,1066,212.0,2,0.05,2025-03-31
259,5259,1066,212.0,2,0.05,2025-03-31



=== VERIFICACIÓN DE UNICIDAD ===
customer_id único: True
product_id único: True
tx_id único: True

=== SHAPES TRAS DEDUPLICAR ===
customers_dedup: (120, 5)
products_dedup: (40, 4)
transactions_dedup: (900, 6)


---
## 4. Nulos

No todos los nulos se resuelven igual:
- a veces imputamos
- a veces eliminamos
- a veces comparamos varias estrategias

### Ejemplo: limpieza básica de `customers`

In [ ]:
customers_clean = customers_dedup.copy()
customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())
customers_clean['city'] = customers_clean['city'].fillna('Unknown')

display(customers_clean.isna().sum())

### Ejercicio 3
Construye `transactions_clean` a partir de `transactions_dedup`:
1. Rellena `discount` con `0.0`.
2. Elimina filas con `product_id` nulo.
3. Comprueba cuántas filas pierdes.

In [5]:
# TODO
# ==========================================
# Construir transactions_clean
# ==========================================

# Copia para no modificar el original
transactions_clean = transactions_dedup.copy()


# ==========================================
# 1. Rellenar discount con 0.0
# ==========================================

transactions_clean['discount'] = (
    transactions_clean['discount']
    .fillna(0.0)
)


# ==========================================
# 2. Eliminar filas con product_id nulo
# ==========================================

rows_before = transactions_clean.shape[0]

transactions_clean = transactions_clean.dropna(
    subset=['product_id']
)

rows_after = transactions_clean.shape[0]


# ==========================================
# 3. Comprobar cuántas filas se pierden
# ==========================================

rows_lost = rows_before - rows_after

print("Filas antes:", rows_before)
print("Filas después:", rows_after)
print("Filas eliminadas:", rows_lost)


# ==========================================
# Verificación de nulos
# ==========================================

print("\nNulos restantes:")
print(transactions_clean[['product_id', 'discount']].isna().sum())

Filas antes: 900
Filas después: 893
Filas eliminadas: 7

Nulos restantes:
product_id    0
discount      0
dtype: int64


### Ejemplo: imputación global frente a imputación por grupo

In [ ]:
customers_clean_global = customers_dedup.copy()
customers_clean_global['age'] = customers_clean_global['age'].fillna(customers_clean_global['age'].median())

customers_clean_group = customers_dedup.copy()
segment_median_age = customers_clean_group.groupby('segment')['age'].transform('median')
customers_clean_group['age'] = customers_clean_group['age'].fillna(segment_median_age)

print('Global median age:', customers_clean_global['age'].median())
print('Group median age by segment:')
display(customers_clean_group.groupby('segment')['age'].median())

### Ejercicio 4
1. Compara nulos restantes y mediana final entre `customers_clean_global` y `customers_clean_group`.
2. Elige una estrategia y guárdala en `customers_clean`.
3. Explica brevemente cuál te parece más defendible.

In [6]:
# TODO

# ==========================================
# 1. Estrategia A: imputación global
#    age -> mediana global
#    city -> 'Unknown'
# ==========================================

customers_clean_global = customers_dedup.copy()

# Rellenar edad con la mediana global
global_median_age = customers_clean_global['age'].median()
customers_clean_global['age'] = (
    customers_clean_global['age']
    .fillna(global_median_age)
)

# Rellenar ciudad con Unknown
customers_clean_global['city'] = (
    customers_clean_global['city']
    .fillna('Unknown')
)


# ==========================================
# 2. Estrategia B: imputación por grupo
#    age -> mediana por segment
#    city -> 'Unknown'
# ==========================================

customers_clean_group = customers_dedup.copy()

# Mediana de edad por segmento (Standard / Premium)
customers_clean_group['age'] = (
    customers_clean_group
    .groupby('segment')['age']
    .transform(lambda s: s.fillna(s.median()))
)

# Rellenar ciudad con Unknown
customers_clean_group['city'] = (
    customers_clean_group['city']
    .fillna('Unknown')
)


# ==========================================
# 3. Comparación de resultados
# ==========================================

print("=== NULOS RESTANTES ===")
print("\nGlobal:")
print(customers_clean_global.isna().sum())

print("\nPor grupo:")
print(customers_clean_group.isna().sum())


print("\n=== MEDIANA FINAL DE AGE ===")
print("Global:", customers_clean_global['age'].median())
print("Por grupo:", customers_clean_group['age'].median())


print("\n=== MEDIANA POR SEGMENTO ===")
print("\nGlobal:")
print(customers_clean_global.groupby('segment')['age'].median())

print("\nPor grupo:")
print(customers_clean_group.groupby('segment')['age'].median())


# ==========================================
# 4. Estrategia elegida
# ==========================================

customers_clean = customers_clean_group.copy()


# ==========================================
# 5. Explicación
# ==========================================

print("""
Estrategia elegida: imputación por segmento.

Motivo:
- La edad puede variar entre clientes Standard y Premium.
- Imputar por grupo conserva mejor las diferencias entre segmentos.
- Es más realista que usar una única mediana global.
- No quedan valores nulos en age ni city.
""")

=== NULOS RESTANTES ===

Global:
customer_id    0
name           0
city           0
segment        0
age            0
dtype: int64

Por grupo:
customer_id    0
name           0
city           0
segment        0
age            0
dtype: int64

=== MEDIANA FINAL DE AGE ===
Global: 41.0
Por grupo: 40.75

=== MEDIANA POR SEGMENTO ===

Global:
segment
Premium     41.0
Standard    41.0
Name: age, dtype: float64

Por grupo:
segment
Premium     41.0
Standard    40.5
Name: age, dtype: float64

Estrategia elegida: imputación por segmento.

Motivo:
- La edad puede variar entre clientes Standard y Premium.
- Imputar por grupo conserva mejor las diferencias entre segmentos.
- Es más realista que usar una única mediana global.
- No quedan valores nulos en age ni city.



---
## 5. Limpieza de texto con `.str`

Operaciones muy frecuentes:
- `strip()`
- `lower()` / `title()`
- reemplazos
- control de dominios

### Ejemplo: normalizar categorías y nombres de producto

In [ ]:
products_clean = products_dedup.copy()
products_clean['category'] = products_clean['category'].astype(str).str.strip().str.title()
products_clean['product_name'] = products_clean['product_name'].astype(str).str.strip().str.title()

print('Category values after cleaning:')
display(products_clean['category'].value_counts())

### Ejercicio 5
1. Limpia `customers_clean['name']` con `.str`.
2. Asegura que `products_clean['category']` solo contiene `{Electronics, Home, Sports, Books}`.
3. Si aparece un valor fuera de dominio, conviértelo a `Unknown`.

In [7]:
# TODO
# ==========================================
# 1. Limpiar customers_clean['name']
# ==========================================

# Eliminar espacios al principio y al final
# Convertir a minúsculas
customers_clean['name'] = (
    customers_clean['name']
    .str.strip()
    .str.lower()
)

# Ejemplo:
# " customer_0 " -> "customer_0"


# ==========================================
# 2. Crear products_clean
# ==========================================

products_clean = products_dedup.copy()

# Limpiar texto de category:
# - quitar espacios
# - poner formato Title Case
products_clean['category'] = (
    products_clean['category']
    .str.strip()
    .str.title()
)


# ==========================================
# 3. Validar dominio permitido
# ==========================================

valid_categories = {
    'Electronics',
    'Home',
    'Sports',
    'Books'
}

# Cualquier categoría fuera del dominio -> 'Unknown'
products_clean['category'] = (
    products_clean['category']
    .where(
        products_clean['category'].isin(valid_categories),
        'Unknown'
    )
)


# ==========================================
# 4. Comprobaciones
# ==========================================

print("=== Nombres limpios ===")
display(customers_clean[['customer_id', 'name']].head())

print("\n=== Categorías únicas ===")
print(products_clean['category'].value_counts())

print("\n=== Valores únicos de category ===")
print(products_clean['category'].unique())

=== Nombres limpios ===


,customer_id,name
0,1000,customer_0
1,1001,customer_1
2,1002,customer_2
3,1003,customer_3
4,1004,customer_4



=== Categorías únicas ===
category
Electronics    16
Home           10
Books           8
Sports          6
Name: count, dtype: int64

=== Valores únicos de category ===
<StringArray>
['Books', 'Electronics', 'Home', 'Sports']
Length: 4, dtype: str


---
## 6. `apply()` cuando aporta valor

`apply` es útil cuando la lógica no es trivial o cuando una regla no sale limpia con una sola operación vectorizada.

### Ejemplo: bandas de descuento

In [ ]:
def discount_band(d):
    if d == 0:
        return 'none'
    if d < 0.10:
        return 'low'
    return 'high'

transactions_clean = transactions_dedup.copy()
transactions_clean['discount'] = transactions_clean['discount'].fillna(0.0)
transactions_clean = transactions_clean.dropna(subset=['product_id']).copy()
transactions_clean['discount_band'] = transactions_clean['discount'].apply(discount_band)

display(transactions_clean['discount_band'].value_counts())

### Ejercicio 6
1. Crea una función `customer_label(row)` con `axis=1`:
   - `VIP_Senior` si `age >= 50` y `segment == 'Premium'`
   - `Premium` si `segment == 'Premium'`
   - `Standard` en otro caso
2. Añade la columna `customer_label` a `customers_clean`.

In [8]:
# TODO
# ==========================================
# 1. Definir función customer_label(row)
# ==========================================

def customer_label(row):
    # VIP_Senior:
    # cliente Premium con edad >= 50
    if row['age'] >= 50 and row['segment'] == 'Premium':
        return 'VIP_Senior'

    # Premium:
    # resto de clientes Premium
    elif row['segment'] == 'Premium':
        return 'Premium'

    # Standard:
    # todos los demás
    else:
        return 'Standard'


# ==========================================
# 2. Crear nueva columna usando apply(axis=1)
# ==========================================

customers_clean['customer_label'] = (
    customers_clean.apply(customer_label, axis=1)
)


# ==========================================
# 3. Comprobación
# ==========================================

print("=== Distribución de customer_label ===")
print(customers_clean['customer_label'].value_counts())

print("\n=== Ejemplos ===")
display(
    customers_clean[
        ['customer_id', 'age', 'segment', 'customer_label']
    ].head(10)
)

=== Distribución de customer_label ===
customer_label
Standard      91
Premium       25
VIP_Senior     4
Name: count, dtype: int64

=== Ejemplos ===


,customer_id,age,segment,customer_label
0,1000,30.0,Standard,Standard
1,1001,35.0,Standard,Standard
2,1002,49.0,Premium,Premium
3,1003,24.0,Standard,Standard
4,1004,42.0,Standard,Standard
5,1005,40.5,Standard,Standard
6,1006,20.0,Standard,Standard
7,1007,41.0,Premium,Premium
8,1008,42.0,Standard,Standard
9,1009,48.0,Premium,Premium


---
## 7. Tipos y validaciones

Antes de unir tablas conviene dejar claras algunas reglas mínimas:
- IDs sin nulos
- cantidades válidas
- descuentos en rango
- precios positivos

### Ejemplo: asserts mínimos

In [ ]:
assert customers_clean['customer_id'].isna().sum() == 0
assert products_clean['product_id'].isna().sum() == 0
assert (products_clean['base_price'].dropna() > 0).all()
assert (transactions_clean['quantity'] >= 1).all()
assert transactions_clean['discount'].between(0, 1).all()

print('Basic validation passed')

### Ejercicio 7
1. Convierte `customer_id` y `product_id` a enteros donde corresponda.
2. Imputa `base_price` con la mediana de la categoría.
3. Añade al menos 3 `assert` útiles sobre las tablas limpias.

In [9]:
# TODO
# ==========================================
# 1. Conversión de tipos (IDs)
# ==========================================

# customer_id -> entero
customers_clean['customer_id'] = (
    customers_clean['customer_id']
    .astype(int)
)

# product_id -> entero (en productos)
products_clean['product_id'] = (
    products_clean['product_id']
    .astype(int)
)

# product_id en transactions_clean -> entero
transactions_clean['product_id'] = (
    transactions_clean['product_id']
    .astype(int)
)


# ==========================================
# 2. Imputar base_price por categoría
# ==========================================

products_clean['base_price'] = (
    products_clean.groupby('category')['base_price']
    .transform(lambda s: s.fillna(s.median()))
)


# ==========================================
# 3. Assertions (validaciones de calidad)
# ==========================================

# 1. No hay nulos en IDs críticos
assert customers_clean['customer_id'].isna().sum() == 0, "Hay nulos en customer_id"
assert products_clean['product_id'].isna().sum() == 0, "Hay nulos en product_id"
assert transactions_clean['product_id'].isna().sum() == 0, "Hay nulos en product_id en transactions"

# 2. IDs deben ser únicos
assert customers_clean['customer_id'].is_unique, "customer_id no es único"
assert products_clean['product_id'].is_unique, "product_id no es único"
assert transactions_clean['tx_id'].is_unique, "tx_id no es único"

# 3. base_price no puede tener nulos tras imputación
assert products_clean['base_price'].isna().sum() == 0, "base_price sigue teniendo nulos"


print("✅ Todas las validaciones pasaron correctamente")

✅ Todas las validaciones pasaron correctamente


---
## 8. `merge` y construcción de `fact_sales`

El objetivo final de esta sesión es construir una tabla analítica a nivel transacción.

### Ejemplo: `left` vs `inner`

In [11]:
demo_left = pd.DataFrame({'id': [1, 2, 3], 'x': [10, 20, 30]})
demo_right = pd.DataFrame({'id': [2, 3, 4], 'y': [200, 300, 400]})

left_join = demo_left.merge(demo_right, on='id', how='left')
inner_join = demo_left.merge(demo_right, on='id', how='inner')

display(left_join)
display(inner_join)

,id,x,y
0,1,10,NaN
1,2,20,200.0
2,3,30,300.0


,id,x,y
0,2,20,200
1,3,30,300


### Ejercicio 8
1. Construye `right_join` y `outer_join` con las tablas demo.
2. Indica cuál conserva más filas.
3. Explica cuándo usarías `left` en un proceso analítico.

In [12]:
# TODO
# ==========================================
# 1. RIGHT JOIN y OUTER JOIN
# ==========================================

# RIGHT JOIN: conserva todas las filas de la derecha (transactions)
right_join = transactions_clean.merge(
    customers_clean,
    on='customer_id',
    how='right'
)

# FULL OUTER JOIN: conserva todo de ambas tablas
outer_join = transactions_clean.merge(
    customers_clean,
    on='customer_id',
    how='outer'
)


# ==========================================
# 2. Comparación de tamaño
# ==========================================

print("Shapes:")
print("Right join:", right_join.shape)
print("Outer join:", outer_join.shape)

print("\nFilas:")
print("Right join filas:", len(right_join))
print("Outer join filas:", len(outer_join))

if len(outer_join) > len(right_join):
    print("\n➡️ El OUTER JOIN conserva más filas.")
else:
    print("\n➡️ El RIGHT JOIN conserva más filas.")


# ==========================================
# 3. Explicación de LEFT JOIN
# ==========================================

print("""
CUÁNDO USAR LEFT JOIN:

Se usa LEFT JOIN cuando tienes una tabla principal (por ejemplo transactions)
y quieres conservar TODAS sus filas, aunque no haya coincidencia en la otra tabla.

Ejemplo típico en análisis:
- Queremos analizar ventas (tabla transactions)
- Aunque algunos clientes o productos estén incompletos o faltantes

Ventajas:
- No pierdes datos del hecho principal (ventas)
- Muy usado en data analytics y business intelligence
- Permite detectar datos faltantes en dimensiones (customers/products)

En resumen:
LEFT JOIN = "quiero mantener mi tabla principal intacta"
""")

Shapes:
Right join: (893, 11)
Outer join: (893, 11)

Filas:
Right join filas: 893
Outer join filas: 893

➡️ El RIGHT JOIN conserva más filas.

CUÁNDO USAR LEFT JOIN:

Se usa LEFT JOIN cuando tienes una tabla principal (por ejemplo transactions)
y quieres conservar TODAS sus filas, aunque no haya coincidencia en la otra tabla.

Ejemplo típico en análisis:
- Queremos analizar ventas (tabla transactions)
- Aunque algunos clientes o productos estén incompletos o faltantes

Ventajas:
- No pierdes datos del hecho principal (ventas)
- Muy usado en data analytics y business intelligence
- Permite detectar datos faltantes en dimensiones (customers/products)

En resumen:
LEFT JOIN = "quiero mantener mi tabla principal intacta"



### Ejercicio 9
Construye `fact_sales`:
1. Une `transactions_clean` con `products_clean` por `product_id` con `how='left'`.
2. Une el resultado con `customers_clean` por `customer_id` con `how='left'`.
3. Crea `amount = quantity * base_price * (1 - discount)`.
4. Devuelve un subconjunto de columnas clave para revisar el resultado.

In [13]:
# TODO
# ==========================================
# 1. LEFT JOIN: transactions + products
# ==========================================

fact_sales = transactions_clean.merge(
    products_clean,
    on='product_id',
    how='left'
)


# ==========================================
# 2. LEFT JOIN: + customers
# ==========================================

fact_sales = fact_sales.merge(
    customers_clean,
    on='customer_id',
    how='left'
)


# ==========================================
# 3. Crear columna amount
# ==========================================

fact_sales['amount'] = (
    fact_sales['quantity'] *
    fact_sales['base_price'] *
    (1 - fact_sales['discount'])
)


# ==========================================
# 4. Selección de columnas clave
# ==========================================

fact_sales = fact_sales[[
    'tx_id',
    'tx_date',
    'customer_id',
    'name',
    'segment',
    'city',
    'product_id',
    'product_name',
    'category',
    'quantity',
    'base_price',
    'discount',
    'amount'
]]


# ==========================================
# 5. Revisión rápida
# ==========================================

print("Shape final:", fact_sales.shape)
display(fact_sales.head())

Shape final: (893, 13)


,tx_id,tx_date,customer_id,name,segment,city,product_id,product_name,category,quantity,base_price,discount,amount
0,5000,2025-02-23,1025,customer_25,Standard,Unknown,219,Product_219,Home,5,64.07,0.00,320.350
1,5001,2025-02-07,1035,customer_35,Standard,Madrid,228,Product_228,Home,3,10.44,0.10,28.188
2,5002,2025-02-17,1000,customer_0,Standard,Valencia,209,Product_209,Books,5,8.35,0.10,37.575
3,5003,2025-01-27,1007,customer_7,Premium,Madrid,233,Product_233,Sports,4,205.01,0.05,779.038
4,5004,2025-01-20,1112,customer_112,Standard,Sevilla,225,Product_225,Home,1,190.46,0.00,190.460


---
## 9. Features mínimas y control final

Una vez creada la fact table, conviene dejar algunas variables analíticas básicas listas para la siguiente sesión.

### Ejemplo: mes y día de la semana

In [14]:
# Este bloque asume que ya existe fact_sales
# fact_sales['month'] = fact_sales['tx_date'].dt.month
# fact_sales['weekday'] = fact_sales['tx_date'].dt.day_name()

### Ejercicio 10
1. Añade a `fact_sales` las columnas `month` y `weekday`.
2. Crea `age_group` con `pd.cut` (`Young`, `Adult`, `Senior`).
3. Construye una tabla `data_quality` con filas, columnas y nulos totales de `customers_clean`, `products_clean`, `transactions_clean` y `fact_sales`.

In [17]:
import pandas as pd

# ==========================================
# 1. Asegurar que fact_sales tiene age
# ==========================================

# Si age no existe, lo recuperamos desde customers_clean
if 'age' not in fact_sales.columns:
    fact_sales = fact_sales.merge(
        customers_clean[['customer_id', 'age']],
        on='customer_id',
        how='left'
    )


# ==========================================
# 2. month y weekday
# ==========================================

fact_sales['tx_date'] = pd.to_datetime(fact_sales['tx_date'])

fact_sales['month'] = fact_sales['tx_date'].dt.month
fact_sales['weekday'] = fact_sales['tx_date'].dt.day_name()


# ==========================================
# 3. age_group con pd.cut
# ==========================================

fact_sales['age_group'] = pd.cut(
    fact_sales['age'],
    bins=[0, 25, 50, 120],
    labels=['Young', 'Adult', 'Senior']
)


# ==========================================
# 4. data_quality summary table
# ==========================================

data_quality = pd.DataFrame({
    'dataset': [
        'customers_clean',
        'products_clean',
        'transactions_clean',
        'fact_sales'
    ],
    'rows': [
        customers_clean.shape[0],
        products_clean.shape[0],
        transactions_clean.shape[0],
        fact_sales.shape[0]
    ],
    'columns': [
        customers_clean.shape[1],
        products_clean.shape[1],
        transactions_clean.shape[1],
        fact_sales.shape[1]
    ],
    'nulls_total': [
        customers_clean.isna().sum().sum(),
        products_clean.isna().sum().sum(),
        transactions_clean.isna().sum().sum(),
        fact_sales.isna().sum().sum()
    ]
})


# ==========================================
# 5. Resultado
# ==========================================

display(fact_sales.head())
display(data_quality)

,tx_id,tx_date,customer_id,name,segment,city,product_id,product_name,category,quantity,base_price,discount,amount,month,weekday,age,age_group
0,5000,2025-02-23,1025,customer_25,Standard,Unknown,219,Product_219,Home,5,64.07,0.00,320.350,2,Sunday,48.0,Adult
1,5001,2025-02-07,1035,customer_35,Standard,Madrid,228,Product_228,Home,3,10.44,0.10,28.188,2,Friday,29.0,Adult
2,5002,2025-02-17,1000,customer_0,Standard,Valencia,209,Product_209,Books,5,8.35,0.10,37.575,2,Monday,30.0,Adult
3,5003,2025-01-27,1007,customer_7,Premium,Madrid,233,Product_233,Sports,4,205.01,0.05,779.038,1,Monday,41.0,Adult
4,5004,2025-01-20,1112,customer_112,Standard,Sevilla,225,Product_225,Home,1,190.46,0.00,190.460,1,Monday,60.0,Senior


,dataset,rows,columns,nulls_total
0,customers_clean,120,6,0
1,products_clean,40,4,0
2,transactions_clean,893,6,0
3,fact_sales,893,17,0
